# Mushroom — eenvoudige voorbereiding

**Bijdrage en AI:** Codex maakte dit concept op verzoek van de gebruiker.
De mapnaam is geen bevestiging van Milans persoonlijke auteurschap of review.
De oorspronkelijke teamoplossing blijft behouden. De kritiek en lesverwijzingen
staan in [REVIEW_FINAL.md](REVIEW_FINAL.md).

Dit is ontwikkelonderzoek. De 900 al bekende ontwikkeltestrecords worden voor
dit experiment niet gescoord. De overblijvende projectdata zijn ook eerder
gebruikt; er ontstaat dus geen nieuwe onafhankelijke eindtest.

## Doel en splits
We behouden de elf bestaande invoervelden. De oorspronkelijke gedeelde vijf folds worden buitenfolds; modelkeuze gebeurt later binnen hun trainingsdeel. Dit voorbereidingsnotebook bevat geen grafieken en fit geen model.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'MushroomFinal/mushroom_project_dataset.csv').exists())
HERE = ROOT/'solutions/milan'
sys.path.insert(0, str(HERE))
from mushroom_alternative import development_data, pipeline, GRID, scores
X, y, ids, folds = development_data()
print('Trainingsrijen:', len(X), '; buitenfolds:', len(folds))
print('Class mapping: e=0, p=1')
print(y.value_counts().sort_index().to_string())
for number, (tr, val) in enumerate(folds, 1):
    assert not set(ids[tr]) & set(ids[val])
    print('Fold', number, ': training', len(tr), '/ validatie', len(val))


Trainingsrijen: 4100 ; buitenfolds: 5
Class mapping: e=0, p=1
class
0    2548
1    1552
Fold 1 : training 3280 / validatie 820
Fold 2 : training 3280 / validatie 820
Fold 3 : training 3280 / validatie 820
Fold 4 : training 3280 / validatie 820
Fold 5 : training 3280 / validatie 820


## Alleen aanwezige velden
De tabel toont het daadwerkelijke modelinvoercontract en ontbrekende waarden binnen de 4.100 ontwikkelingsrijen. Dit is geen dictionary van de volledige UCI-bron.

In [2]:
dictionary = pd.DataFrame({'type': X.dtypes.astype(str), 'missing': X.isna().sum(), 'missing_pct': X.isna().mean()*100, 'unique_known': X.nunique()})
print(dictionary.round(2).to_string())
assert 'jumbled_noise_0' not in X and 'jumbled_noise_1' in X


                      type  missing  missing_pct  unique_known
cap_diameter       float64     1635        39.88          1157
stem_height        float64      846        20.63          1024
stem_width         float64      318         7.76          2019
spore_print_color   object     3888        94.83             7
gill_color          object      798        19.46            12
habitat             object      831        20.27             8
season              object      800        19.51             4
ring_type           object      465        11.34             8
cap_shape           object      331         8.07             7
stem_surface        object     2762        67.37             8
jumbled_noise_1     object      330         8.05             7


## Voorbereiding binnen iedere fit
Numeriek: mediaan. Categorisch: modus of de expliciete categorie missing, gevolgd door one-hot encoding. De volledige transformer zit in de pipeline en wordt dus niet globaal vóór CV gefit. Dit experiment onderzoekt geen proximity-imputatie of automatische verwijdering van nullen.

In [3]:
print(pipeline())
print('Nog niet gefit:', not hasattr(pipeline(), 'classes_'))
assert not hasattr(pipeline(), 'classes_')


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['cap_diameter',
                                                   'stem_height',
                                                   'stem_width']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='most_frequent')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['spore_print_color',
                     